In [1]:
import pandas as pd
import ast
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt


In [ ]:
DATA_PATH = r"C:\Users\nina\Documents\UNI\SEM\thesis\coding\game_color_analysis\data\game_info.csv"

df = pd.read_csv(DATA_PATH)

# Parse complex columns
df["Palette"] = df["Palette"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
df["Genres"] = df["Genres"].apply(
    lambda x: [g["name"] for g in ast.literal_eval(x)] if isinstance(x, str) and x != "Unknown" else []
)
df["Themes"] = df["Themes"].apply(
    lambda x: [t["name"] for t in ast.literal_eval(x)] if isinstance(x, str) and x != "Unknown" else []
)

df.head()

# change genres and themes to ids?
# Palette → [(r,g,b), …]
# Genres → ["Puzzle", "Strategy"]
# Themes → ["Educational"]

,Year,Decade,Game,Screenshot,Genres,Themes,Palette
0,1952,1950,OXO,c:\Users\nina\Documents\UNI\SEM\thesis\coding\...,"[Puzzle, Strategy]",[Educational],"[(44, 44, 44), (147, 147, 147), (79, 79, 79), ..."
1,1952,1950,OXO,c:\Users\nina\Documents\UNI\SEM\thesis\coding\...,"[Puzzle, Strategy]",[Educational],"[(7, 9, 7), (133, 142, 132), (203, 202, 201), ..."
2,1952,1950,OXO,c:\Users\nina\Documents\UNI\SEM\thesis\coding\...,"[Puzzle, Strategy]",[Educational],"[(7, 9, 7), (134, 142, 133), (203, 202, 201), ..."
3,1952,1950,Draughts,c:\Users\nina\Documents\UNI\SEM\thesis\coding\...,[Card & Board Game],[],"[(8, 5, 5), (184, 184, 184), (57, 57, 57), (64..."
4,1954,1950,Pool,c:\Users\nina\Documents\UNI\SEM\thesis\coding\...,[Sport],[],"[(13, 12, 10), (208, 208, 208), (58, 58, 58), ..."


In [4]:
def brightness(color):
    r, g, b = color
    return np.sqrt(0.299*r*r + 0.587*g*g + 0.114*b*b)

def palette_stats(palette):
    if not palette:
        return None
    b = [brightness(c) for c in palette]
    return {
        "mean_brightness": np.mean(b),
        "brightness_var": np.var(b),
        "palette_size": len(palette)
    }


In [9]:
# One row per color
df_colors = (
    df[["Year", "Decade", "Palette"]]
    .explode("Palette")
    .dropna(subset=["Palette"])
    .rename(columns={"Palette": "Color"})
)

# One row per genre
df_genres = (
    df[["Year", "Decade", "Genres"]]
    .explode("Genres")
    .dropna(subset=["Genres"])
    .rename(columns={"Genres": "Genre"})
)

# One row per theme
df_themes = (
    df[["Year", "Decade", "Themes"]]
    .explode("Themes")
    .dropna(subset=["Themes"])
    .rename(columns={"Themes": "Theme"})
)


In [10]:
colors = np.vstack(df_colors["Color"].values)

df_colors["Brightness"] = np.sqrt(
    0.299 * colors[:,0]**2 +
    0.587 * colors[:,1]**2 +
    0.114 * colors[:,2]**2
)


In [11]:
year_colors = (
    df_colors
    .groupby("Year")
    .agg(
        Unique_colors=("Color", "nunique"),
        Mean_brightness=("Brightness", "mean"),
        Brightness_variance=("Brightness", "var")
    )
)

year_top_colors = (
    df_colors
    .groupby("Year")["Color"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)

year_genres = (
    df_genres
    .groupby("Year")
    .agg(Unique_genres=("Genre", "nunique"))
)

year_themes = (
    df_themes
    .groupby("Year")
    .agg(Unique_themes=("Theme", "nunique"))
)

year_top_genres = (
    df_genres
    .groupby("Year")["Genre"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)

year_top_themes = (
    df_themes
    .groupby("Year")["Theme"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)


In [12]:
decade_colors = (
    df_colors
    .groupby("Decade")
    .agg(
        Unique_colors=("Color", "nunique"),
        Mean_brightness=("Brightness", "mean"),
        Brightness_variance=("Brightness", "var")
    )
)

decade_genres = (
    df_genres
    .groupby("Decade")
    .agg(Unique_genres=("Genre", "nunique"))
)

decade_themes = (
    df_themes
    .groupby("Decade")
    .agg(Unique_themes=("Theme", "nunique"))
)


In [14]:
genre_style = (
    df_colors
    .merge(df_genres[["Year", "Decade", "Genre"]], left_index=True, right_index=True, how="left")
    .dropna(subset=["Genre"])
    .groupby("Genre")
    .agg(
        Screenshots=("Color", "count"),
        Unique_colors=("Color", "nunique"),
        Mean_brightness=("Brightness", "mean"),
        Brightness_variance=("Brightness", "var")
    )
    .sort_values("Screenshots", ascending=False)
)

theme_style = (
    df_colors
    .merge(df_themes[["Year", "Decade", "Theme"]], left_index=True, right_index=True, how="left")
    .dropna(subset=["Theme"])
    .groupby("Theme")
    .agg(
        Screenshots=("Color", "count"),
        Unique_colors=("Color", "nunique"),
        Mean_brightness=("Brightness", "mean")
    )
    .sort_values("Screenshots", ascending=False)
)
